# 11 · 健壮性 (维度 6.2) — 重试/熔断/超时 + 运行时 Schema 校验 + 上传硬化

指数退避重试 + 熔断器三态(CLOSED/OPEN/HALF_OPEN) + fallback; jsonschema 运行时校验; 路径穿越/大小/PII脱敏/限流。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from bootstrap import build_controller, status
from services.config import load_settings, load_policy
print('trunk root:', ROOT)

In [ ]:
from services.resilience import Resilient, CircuitOpenError
rt = Resilient(max_retries=2, backoff=0.01, jitter=False, fail_threshold=2, reset_after=30)
n = {'i':0}
def flaky():
    n['i']+=1
    if n['i']<3: raise ConnectionError('transient')
    return 'recovered'
print('retry→', rt.call(flaky, name='x', retry_on=(ConnectionError,)), '| calls=', n['i'])
def down(): raise ConnectionError('down')
for _ in range(2):
    try: rt.call(down, name='cb', retry_on=(ConnectionError,))
    except ConnectionError: pass
print('circuit state=', rt.state('cb'))
try: rt.call(lambda:'never', name='cb')
except CircuitOpenError as e: print('OPEN 快速失败 ✅')
print('fallback on OPEN=', rt.call(lambda:'never', name='cb', fallback=lambda:'offline-kernel'))

In [ ]:
from services import schema_validator as sv
print('valid rfq:', sv.validate_rfq({'material':'6061','quantity':50,'process':'CNC'})['valid'])
bad = sv.validate_rfq({'process':'CNC'})
print('missing required:', bad['valid'], bad['errors'][:2])
print('bad tolerance enum:', sv.validate_rfq({'material':'304','quantity':1,'process':'CNC','tolerance_grade':'IT99'})['valid'])
print('quote negative:', sv.validate_quote({'unit_price':-5})['valid'])

In [ ]:
from services import security as sec
print('path traversal:', sec.safe_filename('../../etc/passwd'))
print('redact key:', sec.redact('my key sk-abc123456789 here'))
print('redact email/phone:', sec.redact('mail alice@x.com tel 13800138000'))
rl = sec.RateLimiter(capacity=2, refill_rate=0.0)
print('rate limit:', [rl.allow('u') for _ in range(3)])
try: sec.size_guard(99*1024*1024)
except ValueError as e: print('size guard:', str(e)[:40])

## 黄金链中的 schema 校验结果 (端到端)

In [ ]:
ctrl = build_controller()
r = ctrl.run(email_text='50 pcs 6061 aluminum, anodizing, IT7.', customer={'name':'Robust Demo'})
print('state=', r['state'], '| schema=', r['schema'])
ctrl.crm.close()